# Deployment: a synchronous endpoint + online features

This is the project's deployment shape. A `Model` gets deployed to a local
SageMaker **endpoint**; the caller looks the block group up in the Feast
online store and invokes the endpoint with that feature vector — one
round-trip, one estimate.

In [ ]:
import boto3
from sagemaker.local import LocalSession
from sagemaker.model import Model

session = LocalSession()
BUCKET = "california-house-pricing"
model = Model(
    image_uri="california-house-pricing:inference",
    model_data="s3://placeholder/model.tar.gz",  # replaced below
    role="arn:aws:iam::111111111111:role/service-role/SageMakerRole",
    sagemaker_session=session,
)

Point `model_data` at the freshest artifact:

In [ ]:
s3 = boto3.client(
    "s3", endpoint_url="http://california-house-pricing-moto:5000"
)
jobs = sorted(
    o["Key"]
    for o in s3.list_objects_v2(Bucket=BUCKET, Prefix="output/").get(
        "Contents", []
    )
    if o["Key"].endswith("model.tar.gz")
)
model.model_data = f"s3://{BUCKET}/{jobs[-1]}"
jobs[-1]

In [ ]:
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="local",
    endpoint_name="california-pricing",
)
predictor.endpoint_name

## Online features → invoke

The production path: entity key in, Feast online features out, endpoint
called once, estimate returned.

In [ ]:
from california_house_pricing.feature_contract import FEATURE_COLUMNS
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo")
row = store.get_online_features(
    features=[
        f"block_group_profile:{c}"
        for c in __import__(
            "california_house_pricing.feature_contract", fromlist=["x"]
        ).FEATURE_COLUMNS
    ],
    entity_rows=[{"block_group_id": "bg-00042"}],
).to_dict()

vector = [row[c][0] for c in FEATURE_COLUMNS]
vector

In [ ]:
import numpy as np

response = predictor.predict(np.array([vector], dtype=float))
response

One `predict` call → one median-value estimate. That is the whole
serving contract; everything else (feature lookup, marshalling) is client
side.

In [ ]:
import numpy as np

# score a second block group straight from the online store
rows = store.get_online_features(
    features=[f"block_group_profile:{c}" for c in FEATURE_COLUMNS],
    entity_rows=[{"block_group_id": f"bg-{i:05d}"} for i in range(5)],
).to_dict()

predictor.predict(
    np.array(
        [[rows[c][i] for c in FEATURE_COLUMNS] for i in range(5)], dtype=float
    )
)